Importar CSV a duckdb, crear una tabla y hacer calculos sencillos

In [1]:
import duckdb

In [2]:
# Creación del "almacén"
con = duckdb.connect("../data/processed/p1.duckdb")

In [3]:
con.sql("DESCRIBE SELECT * FROM '../data/raw/train.csv'").show()

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ store       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ item        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sales       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└─────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘



In [ ]:
# Crear tabla ventas
con.sql("CREATE TABLE ventas AS SELECT * FROM '../data/raw/train.csv'")

In [5]:
# Mostrar tablas creadas
con.sql("SHOW TABLES")

nameVARCHAR
ventas


In [6]:
# Total de filas
con.sql("""
    SELECT 
    COUNT(*) AS TOTAL_FILAS 
    FROM ventas
    """)

TOTAL_FILASBIGINT
913000


In [7]:
# Primera y ultima fecha entre 2013-01-01 y 2017-12-31
con.sql("""
    SELECT
        MIN(date) AS fecha_min,
        MAX(date) AS fecha_max
    FROM ventas
""")

fecha_minDATE,fecha_maxDATE
2013-01-01,2017-12-31


In [8]:
# Cuantas tiendas y cuantos artículos hay
con.sql("""
    SELECT
        COUNT(DISTINCT store) AS num_tiendas,
        COUNT(DISTINCT item) AS num_items
    FROM ventas
""")

num_tiendasBIGINT,num_itemsBIGINT
10,50


In [9]:
con.sql("""
    SELECT
        COUNT(*) AS ventas_nulas_ceros --Count cuenta valores, para contar null uso * para contar filas
    FROM ventas
    WHERE sales IS NULL
        OR sales < 0
""")

ventas_nulas_cerosBIGINT
0


In [26]:
con.sql("""
    SELECT
        DATE_TRUNC('month',date)::DATE AS mes,
        item,
        SUM(sales) AS ventas
    FROM ventas
    GROUP BY 1,2
    ORDER BY 1,2 ASC
""")


mesDATE,itemBIGINT,ventasHUGEINT
2013-01-01,1,3794
2013-01-01,2,10379
2013-01-01,3,6345
2013-01-01,4,3718
2013-01-01,5,3103
2013-01-01,6,10014
2013-01-01,7,10151
2013-01-01,8,13509
2013-01-01,9,8919
2013-01-01,10,12767


In [ ]:
# Comprobacíon: 50 items x 60 meses = 3000 filas ok
con.sql("""
    SELECT
        COUNT(*)
        FROM (
            SELECT
                DATE_TRUNC('month',date) AS mes,
                item,
                SUM(sales) AS ventas
            FROM ventas
            GROUP BY 1,2
            ORDER BY 1,2 ASC) A
""")

count_star()BIGINT
3000


In [36]:
con.sql("""
COPY (    
    WITH mensual AS (
        SELECT 
            DATE_TRUNC('month',date)::DATE AS ds,
            item AS unique_id,
            SUM(sales) AS y
        FROM ventas
        GROUP BY 1,2
        ORDER BY 1,2
        )
    SELECT * FROM mensual
    ) TO '../data/processed/ventas_mensuales.parquet'
""")

CountBIGINT
3000


In [37]:
# Comprobar que se haya guardado correctamente el parquet
con.sql("""
    SELECT COUNT(*)
    FROM '../data/processed/ventas_mensuales.parquet'
""")

count_star()BIGINT
3000


In [ ]:
# Leyendo el parquet
con.sql("""
    SELECT *
    FROM '../data/processed/ventas_mensuales.parquet'
""")

dsDATE,unique_idBIGINT,yDOUBLE
2013-01-01,1,3794.0
2013-01-01,2,10379.0
2013-01-01,3,6345.0
2013-01-01,4,3718.0
2013-01-01,5,3103.0
2013-01-01,6,10014.0
2013-01-01,7,10151.0
2013-01-01,8,13509.0
2013-01-01,9,8919.0
2013-01-01,10,12767.0


In [ ]:
# Pasar el parquet a pandas
df = con.sql("""
    SELECT * FROM '../data/processed/ventas_mensuales.parquet'
    """).df()

In [44]:
df.shape

(3000, 3)

In [47]:
df.dtypes

ds           datetime64[us]
unique_id             int64
y                   float64
dtype: object

In [ ]:
# Cerrar sesión
con.close()